# KidFlix — Convert MKV → MP4 right on your Google Drive

This converts every `.mkv` in your **Kids Media** folder into a browser-
playable `.mp4`, saved next to the original **on your Drive** — no downloading
or re-uploading. It's free and runs in your browser.

**How to use:**
1. Open this notebook in Google Colab: go to https://colab.research.google.com,
   choose **Upload**, and pick this file (`convert_on_drive.ipynb`).
2. Make sure you're signed into the **same Google account** that has the videos
   (brandon.abaki@gmail.com).
3. Run each cell in order with the ▶ button (or Shift+Enter). Approve the
   Drive permission popup when it appears.
4. When it finishes, tell Claude — it'll add the new MP4s to the app.

The video is copied as-is (fast, no quality loss); only audio is re-encoded so
browsers can play it. Files that need it (e.g. HEVC/x265) are fully re-encoded
automatically, which is slower.

In [ ]:
# 1) Connect your Google Drive (approve the popup)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Make sure the converter (ffmpeg) is installed
!apt-get -qq update && apt-get -qq install -y ffmpeg >/dev/null
print('ffmpeg ready')

In [ ]:
# 3) Point at your folder and see what will be converted.
#    If your folder isn't exactly here, edit ROOT to match.
import pathlib

ROOT = '/content/drive/MyDrive/Kids Media'

root = pathlib.Path(ROOT)
assert root.exists(), f'Folder not found: {ROOT}  (check the name/path)'
mkvs = sorted(root.rglob('*.mkv'))
print(f'Found {len(mkvs)} .mkv file(s) under {ROOT}:')
for p in mkvs:
    done = ' (already converted)' if p.with_suffix('.mp4').exists() else ''
    print('  -', p.relative_to(root), done)

In [ ]:
# 4) Convert them all. Safe to re-run — it skips files already done.
import subprocess

def convert(src, dst):
    # Fast path: copy video, re-encode audio to AAC.
    fast = ['ffmpeg','-y','-nostdin','-i',str(src),
            '-map','0:v:0','-map','0:a:0?',
            '-c:v','copy','-c:a','aac','-b:a','192k',
            '-movflags','+faststart','-sn',str(dst)]
    if subprocess.run(fast, capture_output=True, text=True).returncode == 0:
        return 'remuxed'
    # Fallback: re-encode video to H.264 (handles HEVC/x265 etc.). Slower.
    full = ['ffmpeg','-y','-nostdin','-i',str(src),
            '-map','0:v:0','-map','0:a:0?',
            '-c:v','libx264','-crf','21','-preset','fast','-pix_fmt','yuv420p',
            '-c:a','aac','-b:a','192k','-movflags','+faststart','-sn',str(dst)]
    return 're-encoded' if subprocess.run(full).returncode == 0 else 'FAILED'

for i, src in enumerate(mkvs, 1):
    dst = src.with_suffix('.mp4')
    if dst.exists():
        print(f'[{i}/{len(mkvs)}] skip (done): {src.name}')
        continue
    print(f'[{i}/{len(mkvs)}] converting: {src.name} ...')
    result = convert(src, dst)
    print(f'    -> {result}: {dst.name}')

print('\nAll done! The new .mp4 files are in your Drive next to the originals.')